In [ ]:
!pip install datasets

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset

In [ ]:
from pathlib import Path

# Data and checkpoints live in the repo (no Google Drive mount needed).
DATA_DIR = Path("..") / "data" / "raw"
OUTPUT_DIR = Path("..") / "outputs" / "jd_binary"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_DIR =", DATA_DIR.resolve())
print("OUTPUT_DIR =", OUTPUT_DIR.resolve())


In [ ]:
# Sanity check: expected CSV files must exist before training.
train_csv = DATA_DIR / "online_shopping_10_cats.csv"
assert train_csv.exists(), f"Missing training CSV: {train_csv}"
val_csv = DATA_DIR / "updated_comments_small.csv"
assert val_csv.exists(), f"Missing validation CSV: {val_csv}"
print("Found:", train_csv.name, "and", val_csv.name)


In [ ]:
dataset = load_dataset("csv", data_files=str(train_csv), split="train")
dataset = dataset.filter(lambda x: x["review"] is not None)
dataset


In [ ]:

datasets = dataset.train_test_split(test_size=0.1)
datasets

DatasetDict({
    train: Dataset({
        features: ['cat', 'label', 'review'],
        num_rows: 56495
    })
    test: Dataset({
        features: ['cat', 'label', 'review'],
        num_rows: 6278
    })
})

In [ ]:

import torch

tokenizer = AutoTokenizer.from_pretrained("bert-base-chinese")

def process_function(examples):
    tokenized_examples = tokenizer(examples["review"], max_length=128, truncation=True)
    tokenized_examples["labels"] = examples["label"]
    return tokenized_examples

tokenized_datasets = datasets.map(process_function, batched=True, remove_columns=datasets["train"].column_names)
tokenized_datasets

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/56495 [00:00<?, ? examples/s]

Map:   0%|          | 0/6278 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 56495
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 6278
    })
})

In [ ]:


model = AutoModelForSequenceClassification.from_pretrained("bert-base-chinese")



Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
!pip install evaluate

In [ ]:
import evaluate

acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
auc_metric = evaluate.load("roc_auc")

In [ ]:
def eval_metric(eval_predict):
    predictions, labels = eval_predict

    # 如果预测结果是多个类别的概率，取正类的概率
    if predictions.ndim > 1 and predictions.shape[1] > 1:
        probabilities = predictions[:, 1]
    else:
        probabilities = predictions

    # 根据概率计算预测类别
    predicted_classes = probabilities.argmax(axis=-1) if probabilities.ndim > 1 else (probabilities >= 0.5).astype(int)

    # 计算各项指标
    acc = acc_metric.compute(predictions=predicted_classes, references=labels)
    f1 = f1_metric.compute(predictions=predicted_classes, references=labels)
    auc = auc_metric.compute(prediction_scores=probabilities, references=labels)

    # 更新字典，将F1得分和AUC添加进去
    acc.update(f1)
    acc["auc"] = auc["roc_auc"]

    return acc

In [ ]:
!pip install transformers[torch]

In [ ]:
import sys
print(sys.path)

['/content', '/env/python', '/usr/lib/python310.zip', '/usr/lib/python3.10', '/usr/lib/python3.10/lib-dynload', '', '/usr/local/lib/python3.10/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.10/dist-packages/IPython/extensions', '/usr/local/lib/python3.10/dist-packages/setuptools/_vendor', '/root/.ipython', '/root/.cache/huggingface/modules']


In [ ]:
!pip install accelerate -U

In [ ]:
from transformers import TrainingArguments

In [ ]:
pip install transformers[torch]

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

In [ ]:
pip install accelerate

In [ ]:
!pip install pytorch-transformers

In [ ]:
!pip install transformers[torch]
!pip install accelerate -U

In [ ]:
train_args = TrainingArguments(output_dir=str(OUTPUT_DIR), # 输出文件夹
                               per_device_train_batch_size=128,  # 训练时的batch_size
                               per_device_eval_batch_size=128,  # 验证时的batch_size
                               logging_steps=10,                # log 打印的频率
                               evaluation_strategy="epoch",     # 评估策略
                               save_strategy="epoch",           # 保存策略
                               save_total_limit=5,              # 最大保存数
                               learning_rate=2e-5,              # 学习率
                               num_train_epochs=5.0,
                               weight_decay=0.01,               # weight_decay
                               metric_for_best_model="f1",      # 设定评估指标
                               load_best_model_at_end=True)     # 训练完成后加载最优模型
train_args


In [ ]:
!pip install --upgrade transformers
!pip install --upgrade datasets

In [ ]:
from transformers import DataCollatorWithPadding
trainer = Trainer(model=model,
                  args=train_args,
                  train_dataset=tokenized_datasets["train"],
                  eval_dataset=tokenized_datasets["test"],
                  data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
                  compute_metrics=eval_metric)

In [ ]:
trainer.evaluate(tokenized_datasets["test"]) ###訓練前

{'eval_loss': 0.7254841923713684,
 'eval_model_preparation_time': 0.0061,
 'eval_accuracy': 0.4863013698630137,
 'eval_f1': 0.0,
 'eval_auc': 0.5791167919723135,
 'eval_runtime': 40.6039,
 'eval_samples_per_second': 154.616,
 'eval_steps_per_second': 1.231}

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset

In [ ]:
trainer.train() ###訓練中

Epoch,Training Loss,Validation Loss,Model Preparation Time,Accuracy,F1,Auc
1,0.147700,0.147815,0.006100,0.949028,0.949606,0.985583
2,0.120600,0.139237,0.006100,0.950303,0.950382,0.986715
3,0.050500,0.161408,0.006100,0.955400,0.955682,0.985896
4,0.054100,0.171747,0.006100,0.956037,0.956740,0.984149
5,0.047700,0.181023,0.006100,0.956833,0.957570,0.982703


TrainOutput(global_step=2210, training_loss=0.0985027116280875, metrics={'train_runtime': 5692.4295, 'train_samples_per_second': 49.623, 'train_steps_per_second': 0.388, 'total_flos': 1.8580573840704e+16, 'train_loss': 0.0985027116280875, 'epoch': 5.0})

In [ ]:
trainer.evaluate(tokenized_datasets["test"])

{'eval_loss': 0.1810225546360016,
 'eval_model_preparation_time': 0.0061,
 'eval_accuracy': 0.956833386428799,
 'eval_f1': 0.9575700641928918,
 'eval_auc': 0.9827030979821603,
 'eval_runtime': 43.9311,
 'eval_samples_per_second': 142.905,
 'eval_steps_per_second': 1.138,
 'epoch': 5.0}

In [ ]:
trainer.predict(tokenized_datasets["test"])

PredictionOutput(predictions=array([[ 0.9133185 , -0.57714057],
       [ 2.5669408 , -2.4198813 ],
       [ 4.1322365 , -3.3631067 ],
       ...,
       [-3.6179278 ,  3.1932182 ],
       [ 2.9208598 , -2.5688632 ],
       [ 3.1592648 , -2.7961507 ]], dtype=float32), label_ids=array([1, 0, 0, ..., 1, 0, 0]), metrics={'test_loss': 0.1810225546360016, 'test_model_preparation_time': 0.0061, 'test_accuracy': 0.956833386428799, 'test_f1': 0.9575700641928918, 'test_auc': 0.9827030979821603, 'test_runtime': 43.3989, 'test_samples_per_second': 144.658, 'test_steps_per_second': 1.152})

In [ ]:
validation_dataset = load_dataset(
    "csv",
    data_files=str(DATA_DIR / "updated_comments_small.csv"),
    split="train",
)

# Drop rows without a review field
validation_dataset = validation_dataset.filter(lambda x: x["review"] is not None)


In [ ]:
# 使用同樣的 tokenizer 進行分詞
def process_validation_function(examples):
    tokenized_examples = tokenizer(examples["review"], max_length=128, truncation=True)
    tokenized_examples["labels"] = examples["label"]
    return tokenized_examples

# 對小資料集進行分詞處理
tokenized_validation_dataset = validation_dataset.map(process_validation_function, batched=True, remove_columns=validation_dataset.column_names)

Map:   0%|          | 0/658 [00:00<?, ? examples/s]

In [ ]:
# Use a padding token to ensure all sequences in a batch are the same length
def process_validation_function(examples):
    tokenized_examples = tokenizer(
        examples["review"],
        max_length=128,
        truncation=True,
        padding='max_length'  # Add padding
    )
    tokenized_examples["labels"] = examples["label"]
    return tokenized_examples

# Apply the updated processing function
tokenized_validation_dataset = validation_dataset.map(
    process_validation_function,
    batched=True,
    remove_columns=validation_dataset.column_names
)

Map:   0%|          | 0/658 [00:00<?, ? examples/s]

In [ ]:
predictions = trainer.predict(tokenized_validation_dataset) ###訓練後灌入dcard驗證集
# The predict method expects the entire Dataset object
print(predictions) # Access the predictions from the output

PredictionOutput(predictions=array([[ 0.5096891, -0.2808244],
       [ 3.1754367, -2.8716924],
       [ 2.6836712, -2.2337787],
       ...,
       [ 2.4193301, -1.7675536],
       [ 4.009645 , -3.3413947],
       [ 3.048748 , -2.2697704]], dtype=float32), label_ids=array([0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0,
       0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0,
       0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0,
       1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0,
       0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0,
       1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0,
       1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0,
       0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1,
       1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0,
    